In [6]:
import pandas as pd
import sqlite3

In [7]:
# Шляхи до файлів
paths = {
    "customers": "/Users/avramenko.a.m/Development/Project#1/practice/datasets/archive/olist_customers_dataset.xlsx",
    "geolocation": "/Users/avramenko.a.m/Development/Project#1/practice/datasets/archive/olist_geolocation_dataset.xlsx",
    "order_items": "/Users/avramenko.a.m/Development/Project#1/practice/datasets/archive/olist_order_items_dataset.xlsx",
    "order_payments": "/Users/avramenko.a.m/Development/Project#1/practice/datasets/archive/olist_order_payments_dataset.xlsx",
    "order_reviews": "/Users/avramenko.a.m/Development/Project#1/practice/datasets/archive/olist_order_reviews_dataset.xlsx",
    "orders": "/Users/avramenko.a.m/Development/Project#1/practice/datasets/archive/olist_orders_dataset.xlsx",
    "products": "/Users/avramenko.a.m/Development/Project#1/practice/datasets/archive/olist_products_dataset.xlsx",
    "sellers": "/Users/avramenko.a.m/Development/Project#1/practice/datasets/archive/olist_sellers_dataset.xlsx",
    "category_translation": "/Users/avramenko.a.m/Development/Project#1/practice/datasets/archive/product_category_name_translation.xlsx",
}

dataframes = {}


In [8]:
# 1) Підключення (in-memory). Якщо хочеш зберігати базу на диск — заміни на "olist.db"
conn = sqlite3.connect(":memory:")

In [9]:
dataframes = {name: pd.read_excel(path) for name, path in paths.items()}

In [10]:
for name, path in paths.items():
    dataframes[name].to_sql(name, conn, index=False, if_exists="replace")


## Дослідницький Аналіз Данних

### Блок 1 — Загальний огляд бізнесу
1. Скільки всього замовлень у датасеті?\
2. Скільки унікальних клієнтів?
3. Скільки продавців працює на платформі?
4. Скільки унікальних товарів продається?
5. Який середній чек замовлення?

### Блок 2 — Динаміка продажів
6. Як змінювалась кількість замовлень по місяцях?
7. Чи є сезонність у продажах?
8. Який місяць має найбільше продажів?
9. Як змінюється GMV (Gross Merchandise Value) з часом?

### Блок 3 — Поведінка клієнтів
10. Скільки замовлень робить середній клієнт?
11. Який % клієнтів роблять повторні покупки?
12. Який середній час між замовленнями?
13. Який Customer Lifetime Value (CLV)?

### Блок 4 — Аналіз платежів
14. Які способи оплати найпопулярніші?
15. Який середній чек по кожному способу оплати?
16. Який дохід по кожному способу оплати?

### Блок 5 — Відгуки клієнтів
17. Який середній рейтинг замовлень?
18. Яка частка негативних відгуків (1–2)?
19. Чи впливає час доставки на рейтинг?
20. Які категорії товарів отримують найгірші відгуки?

### Блок 6 — Логістика
21. Скільки в середньому триває доставка?
22. Чи є затримки доставки?
23. Які штати мають найбільші затримки?
24. Чи впливають затримки на рейтинг?

### Блок 7 — Аналіз продавців
25. Скільки замовлень у середнього продавця?
26. Хто топ-10 продавців за кількістю продажів?
27. Чи є залежність між рейтингом і продавцем?
28. Які продавці мають найбільше негативних відгуків?

### Блок 8 — Аналіз товарів
29. Які категорії товарів продаються найкраще?
30. Які категорії генерують найбільший дохід?
31. Які категорії мають найгірші відгуки?
32. Які категорії мають найдовшу доставку?

### Блок 9 — Географічний аналіз
33. В яких штатах найбільше покупців?
34. В яких штатах найбільше продавців?
35. Чи є різниця в середньому чеку між регіонами?
36. Чи є залежність між відстанню та часом доставки?

### Блок 10 — Ризики для бізнесу
37. Яка частка скасованих замовлень?
38. Чи пов’язані скасування з певними продавцями?
39. Чи впливає довга доставка на скасування?
40. Які категорії мають найбільше проблем?

## Блок 1 — Загальний огляд бізнесу
1. Скільки всього замовлень у датасеті?\
2. Скільки унікальних клієнтів?
3. Скільки продавців працює на платформі?
4. Скільки унікальних товарів продається?
5. Який середній чек замовлення?

In [11]:
# 1. Скільки всього замовлень у датасеті?
query = """
SELECT
    COUNT(DISTINCT(order_id))  AS cnt_orders
FROM
    orders
WHERE order_id IS NOT NULL
LIMIT 10;
"""
pd.read_sql_query(query, conn)

,cnt_orders
0,99441


In [12]:
# 2. Скільки унікальних клієнтів ?
query = """
SELECT
    COUNT(DISTINCT(customer_id)) AS uniq_customer
FROM
    orders
WHERE customer_id IS NOT NULL
LIMIT 10;
"""
pd.read_sql_query(query, conn)

,uniq_customer
0,99441


In [13]:
# 3. Скільки продавців працює на платформі?
query = """
SELECT
    COUNT(DISTINCT(seller_id)) AS cnt_sellers
FROM
    sellers
WHERE seller_id IS NOT NULL
LIMIT 10;
"""
pd.read_sql_query(query, conn)

,cnt_sellers
0,3095


In [14]:
# 4. Скільки унікальних товарів продається?
query = """
SELECT
    COUNT(DISTINCT(product_category_name)) AS cnt_uniq_product
FROM
    products
LIMIT 10;
"""
pd.read_sql_query(query, conn)


,cnt_uniq_product
0,73


In [15]:
# 5. Який середній чек замовлення?
query = """
SELECT
    ROUND(AVG(payment_value)) AS avg_check
FROM
    order_payments
WHERE payment_value IS NOT NULL
LIMIT 10;
"""
pd.read_sql_query(query, conn)

,avg_check
0,170.0


## Блок 2 — Динаміка продажів
6. Як змінювалась кількість замовлень по місяцях?
7. Чи є сезонність у продажах?
8. Який місяць має найбільше продажів?
9. Як змінюється GMV (Gross Merchandise Value) з часом?

In [16]:
# 6. Як змінювалась кількість замовлень по місяцях?
query = """
SELECT
    strftime('%Y-%m', order_purchase_timestamp) AS order_month,
    COUNT(*) AS orders_count
FROM
    orders
GROUP BY order_month 
ORDER BY order_month;
"""
pd.read_sql_query(query, conn)

,order_month,orders_count
0,2016-09,4
1,2016-10,324
2,2016-12,1
3,2017-01,800
4,2017-02,1780
5,2017-03,2682
6,2017-04,2404
7,2017-05,3700
8,2017-06,3245
9,2017-07,4026


### Висновок:
Кількість замовлень швидко зростала протягом 2017 року, збільшившись майже вдесятеро з січня по листопад.
Чіткий пік спостерігається в листопаді 2017 року, ймовірно, завдяки розпродажам у Чорну п'ятницю.
У 2018 році платформа досягла стабільного рівня попиту, а щомісячна кількість замовлень коливалася від 6500 до 7200.
Дані за вересень та жовтень 2018 року видаються неповними та не повинні враховуватися в аналізі тенденцій.

In [17]:
# 7. Чи є сезонність у продажах?
query = """
SELECT
    substr(year_month, 6, 2) AS month,
    ROUND(AVG(month_orders),2) AS avg_orders
FROM
(
    SELECT
        strftime('%Y-%m', order_purchase_timestamp) AS year_month,
        COUNT(*) AS month_orders
    FROM orders
    GROUP BY year_month
    HAVING COUNT(*) > 1000
)
GROUP BY month
ORDER BY month;
"""
pd.read_sql_query(query, conn)

,month,avg_orders
0,01,7269.0
1,02,4254.0
2,03,4946.5
3,04,4671.5
4,05,5286.5
5,06,4706.0
6,07,5159.0
7,08,5421.5
8,09,4285.0
9,10,4631.0


### Висновок: 
1) Аналіз виявляє сезонні закономірності в продажах. 
2) Обсяги замовлень досягають піку в листопаді, ймовірно, через акції Чорної п'ятниці, і залишаються відносно високими в грудні через святкові покупки. 
3) Продажі, як правило, нижчі в лютому та вересні, тоді як середина року демонструє відносно стабільний попит.

In [18]:
# 8. Який місяць має найбільше продажів?
query = """
SELECT
    substr(year_month, 6, 2) AS month,
    ROUND(AVG(month_orders),2) AS avg_orders
FROM
(
    SELECT
        strftime('%Y-%m', order_purchase_timestamp) AS year_month,
        COUNT(*) AS month_orders
    FROM orders
    GROUP BY year_month
    HAVING COUNT(*) > 1000
)
GROUP BY month
ORDER BY avg_orders DESC
LIMIT 1;
"""
pd.read_sql_query(query, conn)

,month,avg_orders
0,11,7544.0


### Висновок:
Обсяги замовлень досягають піку в листопаді, ймовірно, через акції Чорної п'ятниці.  


In [19]:
# 9. Як змінюється GMV (Gross Merchandise Value) з часом?
query = """
SELECT 
    strftime('%Y-%m', o.order_purchase_timestamp) AS year_month,
    ROUND(SUM(oi.price + oi.freight_value),2) AS gmv
    /* Сума GMV */
FROM
    orders o
JOIN order_items oi
ON o.order_id = oi.order_id
GROUP BY year_month
ORDER BY year_month; 
"""
pd.read_sql_query(query, conn)

,year_month,gmv
0,2016-09,354.75
1,2016-10,129152.32
2,2016-12,19.62
3,2017-01,300048.79
4,2017-02,690132.64
5,2017-03,1217775.61
6,2017-04,1105590.30
7,2017-05,2158256.48
8,2017-06,2010578.18
9,2017-07,2251571.67


### Висновок:
1) Жовтень 2016 року був не поганий як для почтаку роботи маркетплейса
2) Можно спостерігати дуже сильне зростання в 2017. GMV росте майже кожен місяць ~15х зростання за рік. 
2) Можно спостерігати передсезонну просадку в вересні 2017 року
3) Пік припадає на листопад 2017 року, ймовірна причина Black Friday. 

## Блок 3 — Поведінка клієнтів
10. Скільки замовлень робить середній клієнт?
11. Який % клієнтів роблять повторні покупки?
12. Який середній час між замовленнями?
13. Який Customer Lifetime Value (CLV)?

In [20]:
# 10. Скільки замовлень робить середній клієнт?
query = """
SELECT
    ROUND(AVG(cnt_orders), 2) AS avg_ord_per_customer
FROM
(
SELECT
    customer_unique_id,
    COUNT(DISTINCT order_id) AS cnt_orders -- Кількість замовлень на кожного клієнта
FROM customers c
JOIN orders o
ON c.customer_id = o.customer_id
GROUP BY customer_unique_id
)
"""
pd.read_sql_query(query, conn)


,avg_ord_per_customer
0,1.03


### Висновок:
Середня кількість замовлень на одного клієнта становить приблизно 1,03, що свідчить про те, що більшість клієнтів розміщують лише одне замовлення на платформі і про відносно низький рівень утримання клієнтів.

In [21]:
# 11. Який % клієнтів роблять повторні покупки?
query = """
SELECT
    ROUND(
        100.0 * SUM(CASE WHEN cnt_orders > 1 THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS repeat_purchase_rate
FROM
(
SELECT
    customer_unique_id,
    COUNT(DISTINCT order_id) AS cnt_orders
FROM customers c
JOIN orders o 
ON c.customer_id = o.customer_id
GROUP BY customer_unique_id
)
"""
pd.read_sql_query(query, conn)



,repeat_purchase_rate
0,3.12


### Висновок: 
Рівень повторних покупок дуже низький, що свідчить про те, що більшість клієнтів здійснюють лише одну покупку на платформі. Це говорить про низький рівень утримання клієнтів.

In [22]:
# 12. Який середній час між замовленнями?
query = """
SELECT
    ROUND(AVG(days_between_orders),2) AS avg_days_between_orders
FROM
(
SELECT
    customer_unique_id,
    julianday(order_purchase_timestamp) -
    julianday(LAG(order_purchase_timestamp)
        OVER (
            PARTITION BY customer_unique_id
            ORDER BY order_purchase_timestamp
        )
    ) AS days_between_orders
FROM customers c
JOIN orders o
ON c.customer_id = o.customer_id
)
WHERE days_between_orders IS NOT NULL;
""" 
pd.read_sql_query(query, conn)


,avg_days_between_orders
0,78.23


### Висновок: 
Середній час між повторними покупками становить приблизно 78 днів, що свідчить про те, що постійні клієнти зазвичай розміщують наступне замовлення приблизно через два-три місяці. Це говорить про відносно довгий цикл покупки та підкреслює можливості для маркетингових стратегій утримання клієнтів.


In [23]:
# 13. Який Customer Lifetime Value (CLV)?
query = """
SELECT
    AVG(customer_revenue) AS avg_clv
FROM (
    SELECT
        c.customer_unique_id, 
        SUM(oi.price + oi.freight_value) AS customer_revenue
    FROM
        orders o    
    JOIN customers c 
        ON o.customer_id = c.customer_id
    JOIN order_items oi 
         ON oi.order_id = o.order_id
    GROUP BY c.customer_unique_id
);
"""
pd.read_sql_query(query, conn)

,avg_clv
0,502.053141


### Висновок: 
1) Верхня межа витрати на маркетинг 100-300 якщо більше, на межі збитковості. 
2) CLV ≈ один чек
3) Бізнес залежить від нових клієнтів (Growth = Acquisition, а не retention)
- Постійсна залежність від реклами
- Низька лояльність
4) Найбільша можливість росту в збільшені повторних покупок

### Рекомендації:
1) Щоб покращіти CLV рекомендовано більше впроваджувати:
- email маркетинг
- персональні рекомендації
- бонуси за повторні покупки 
- підписки

## Блок 4 — Аналіз платежів
14. Які способи оплати найпопулярніші?
15. Який середній чек по кожному способу оплати?
16. Який дохід по кожному способу оплати?
 

In [24]:
# 14. Які способи оплати найпопулярніші?
query = """
SELECT
    payment_type,
    COUNT(payment_type) AS cnt_type,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 2) AS percent 
FROM
    order_payments
WHERE payment_type IS NOT NULL
GROUP BY payment_type
ORDER BY cnt_type DESC;
"""
pd.read_sql_query(query, conn)

,payment_type,cnt_type,percent
0,credit_card,76795,73.92
1,boleto,19784,19.04
2,voucher,5775,5.56
3,debit_card,1529,1.47
4,not_defined,3,0.00


### Висновок:
1) Основний драйвер - credit card. Бізнес залежить від:
- платіжних шлюзів
- стабільних транзакцій
- комісій банків
2) boleto 
- це локальна специфіка Бразилії
- частина користувачів НЕ використовує карт
3) Voucher
- маркетинг вже працює
- є сегмент користвувачів, які реагують на знижки 
4) Debit card
- UX гірший ніж у credit card
- користвувачі не довіряють
- або просто не популярне

### Рекомендації:
1) Опитимізовувати credit card flow
- швидкість оплати
- мінімум відмов
- User Expirience (UX)
2) Розвивати альтернативи:
- boleto
- Apple Pay/ Google Pay
3) Використовувати voucher
- підвищувати retantion
- стимулювати повторні покупки


In [25]:
# 15. Який середній чек по кожному способу оплати?
query = """
SELECT
    payment_type,
    ROUND(AVG(order_total), 2) AS avg_order_value
FROM (
    SELECT
        order_id,
        payment_type,
        SUM(payment_value) AS order_total
    FROM
        order_payments
    GROUP BY order_id, payment_type
)
GROUP BY payment_type
ORDER BY avg_order_value DESC;
 """
pd.read_sql_query(query, conn)

,payment_type,avg_order_value
0,voucher,236.56
1,credit_card,175.91
2,boleto,155.85
3,debit_card,149.22
4,not_defined,0.00


### Висновок:
1) У Voucher найвищій середній чек але не найбільший дохід це дуже важливо 
- знижки симулюють більше покупок
- клієнти "докуповують", щоб використовувати бонус
- маркетинг працює 
2) Credit Card - стабільний сегмент
3) Boleto s Debit - нижчій середній чек → дешевші покупки
- вірогідно більш чутливі до ціни клієнти
- менша платіжоспроможність
- або інший тип аудіторії
Ключовий бізнес-інсайт: Voucher збільшує середній чек → це інструмент для росту retention та revenue

### Рекомендації:
1) Маштабувати Voucher
- акції
- персональні знажки
- бонуси за великі покупки
2) Оптимізація Credit Card
- це ядро бізнесу
- тут найбільний обʼєм і дохід
3) Преревірити маржу:
- чи не зʼїдають знижки прибуток ?

In [26]:
# 16. Який дохід по кожному способу оплати?
query = """
SELECT
    payment_type,
    ROUND(SUM(order_total), 2) AS revenue_pay_type
FROM (
    SELECT
        order_id,
        payment_type,
        SUM(payment_value) AS order_total
    FROM
        order_payments
    GROUP BY order_id, payment_type
)
GROUP BY payment_type
ORDER BY revenue_pay_type DESC;
 """
pd.read_sql_query(query, conn)

,payment_type,revenue_pay_type
0,credit_card,13458148.60
1,boleto,3083347.53
2,voucher,914534.33
3,debit_card,228001.41
4,not_defined,0.00


### Висновок:
Кредитні картки генерують більшу частину загального доходу, що робить їх основним методом оплати для бізнесу. Хоча оплата ваучерами має найвищу середню вартість замовлення, їхній загальний внесок у дохід є відносно невеликим через нижчу частоту використання. Це підкреслює розрив між розміром транзакції та загальним впливом на дохід. Boleto залишається важливим вторинним методом оплати, тоді як використання дебетових карток мінімальне.

## Блок 5 — Відгуки клієнтів
17. Який середній рейтинг замовлень?
18. Яка частка негативних відгуків (1–2)?
19. Чи впливає час доставки на рейтинг?
20. Які категорії товарів отримують найгірші відгуки?

In [27]:
# 17. Який середній рейтинг замовлень?
query = """
SELECT
    ROUND(AVG(review_score), 2) AS avg_review_score
FROM 
    order_reviews
WHERE review_score IS NOT NULL;
 """
pd.read_sql_query(query, conn)

,avg_review_score
0,4.09


In [28]:
# 18. Яка частка негативних відгуків (1–2)?
query = """
SELECT
    SUM(percent) AS part_reviews_score_1_and_2
FROM
    (
    SELECT
        review_score,
        COUNT(review_score) AS cnt_review_score,
        ROUND(100.0 * COUNT(*)/ SUM(COUNT(*))  OVER(), 2) AS percent
    FROM 
        order_reviews
    WHERE
       review_score IS NOT NULL
    GROUP BY review_score
    ORDER BY review_score
    )
WHERE review_score <= 2 
 """
pd.read_sql_query(query, conn)

,part_reviews_score_1_and_2
0,14.69


### Висновок:
1) Головний інсайт: 
- Високий середній рейтинг маскує 15% негативу

### Рекомендації:
1) Аналіз негативу (ТОП пріорітет):
- категорії ()
- покращіти доставку
- payment ()
- фокус на (1-2) це найцінніші дані для покращення 
- зменшити негатив хочаб до 10%

In [29]:
# 19. Чи впливає час доставки на рейтинг?
query = """
SELECT
    review_score,
    AVG(delivery_time) AS avg_delivery_time
FROM
    (
    SELECT
        julianday(order_delivered_customer_date) - julianday(order_approved_at) AS delivery_time,
        review_score  
    FROM 
        orders o
    JOIN order_reviews orr ON o.order_id = orr.order_id
    WHERE
        order_delivered_customer_date IS NOT NULL
        AND order_approved_at IS NOT NULL
    )
GROUP BY review_score
"""
pd.read_sql_query(query, conn)

,review_score,avg_delivery_time
0,1,20.856306
1,2,16.204752
2,3,13.808944
3,4,11.881622
4,5,10.269995


### Висновок: 
1) Головний інсайт чим довша доставка тим нижче рейтинг
2) Доставка - один із ключових фаткорів задоволеності клієнтів

### Рекомендації: 
1) Оптимізовувати доставку
- логістика
- швидкість обрабки замовлень
- склади
2) Контролюваьти SLA (Service Level Agreement) 
- target: < 12 дні (після цього рейтинги різко ростуть)
3) Прогноз затримок
- попереджати клієнтів
- показувати реальний ETA (Estimated Time of Arrival) 
4) Пріоритезувати повільні регіони

In [30]:
# 20. Які категорії товарів отримують найгірші відгуки?
query = """
SELECT
    ct.product_category_name_english,
    COUNT(*) AS review_count,
    ROUND(AVG(review_score), 2) AS avg_review_score
FROM
    products p
JOIN order_items oi
    ON p.product_id = oi.product_id
JOIN order_reviews orr
    ON oi.order_id = orr.order_id 
JOIN category_translation ct 
    ON p.product_category_name = ct.product_category_name
GROUP BY ct.product_category_name_english
HAVING COUNT(*) >= 30
ORDER BY avg_review_score ASC, review_count DESC
LIMIT 20;
"""
pd.read_sql_query(query, conn)


,product_category_name_english,review_count,avg_review_score
0,diapers_and_hygiene,39,3.26
1,office_furniture,1687,3.49
2,fashion_male_clothing,131,3.64
3,fixed_telephony,262,3.68
4,party_supplies,43,3.77
5,fashio_female_clothing,50,3.78
6,furniture_mattress_and_upholstery,38,3.82
7,home_confort,435,3.83
8,audio,361,3.83
9,construction_tools_safety,193,3.84


### Висновок:
1) Критичні категорії:
- office_furniture (Count: 1687, avg_review: 3.49)
- home_confort (Count: 435, avg_review: 3.83)
- audio (Count: 361, avg_review: 3.83)
Малий рейтинг + великий обʼєм = великий вплив на середню оцінку
2) Малі категорії (менше значення)
Шум а не проблема:
- diapers_and_hygiene (39 відгуків)
- party_supplies (43 відгуки)
3) Проблема сконцентрована в меблях і складних товарах:
- office_furniture
- furniture_*
- home_*
Ймовірна причина:
- Довга доставка товарів
- Пошкодження (Великі товари, Ризик транспортування)
- Очікування VS реальність (фото != реальність, Складність товарів)

### Рекомендації:
1) Фокус на office_furniture (Це найбільша точка болю)
- аналіз доставки
- аналіз поверненя
- аналіз review текстів
2) Покращіти логістику
- зменшити delivery_time
3) Контент товарі
- більше фото
- точні розміри
- відео/інструкції 
4) Сегментуванняч проблеми
- перевірити кореляцію між часом достави, категорією, оцінкою, кількітсь оцінок. 


## Блок 6 — Логістика
21. Скільки в середньому триває доставка?
22. Чи є затримки доставки?
23. Які штати мають найбільші затримки?
24. Чи впливають затримки на рейтинг?

In [31]:
# 21. Скільки в середньому триває доставка?
query = """
SELECT
    AVG(julianday(order_delivered_customer_date) - julianday(order_approved_at)) AS delivery_time
FROM
    orders
"""
pd.read_sql_query(query, conn)



,delivery_time
0,12.130357


In [32]:
# 22. Чи є затримки доставки?
query = """
SELECT
    ROUND(
        100.0 * SUM(
            CASE 
                WHEN julianday(order_delivered_customer_date)
                    > julianday(order_estimated_delivery_date)
                THEN 1 ELSE 0
            END
        ) / COUNT(*), 2
    ) AS delayed_percent
FROM
    orders
WHERE 
    order_delivered_customer_date IS NOT NULL
    AND order_estimated_delivery_date IS NOT NULL
"""
pd.read_sql_query(query, conn)


,delayed_percent
0,8.11


### Висновок:
1) 8.1 % замолень були доставлені невчасно

In [33]:
# 23. Які штати мають найбільші затримки?
query = """
SELECT
    c.customer_state,
    ROUND(AVG(julianday(o.order_estimated_delivery_date)- julianday(o.order_delivered_customer_date)), 2 ) AS avg_delay_days,
    COUNT(*) AS orders_count  
FROM 
    orders o
JOIN customers c ON o.customer_id = c.customer_id
WHERE
    o.order_delivered_customer_date IS NOT NULL
    AND o.order_estimated_delivery_date IS NOT NULL
GROUP BY c.customer_state
ORDER BY avg_delay_days DESC;
"""
pd.read_sql_query(query, conn)


,customer_state,avg_delay_days,orders_count
0,AC,20.08,80
1,RO,19.40,243
2,AP,19.06,67
3,AM,18.85,145
4,RR,16.59,41
5,MT,13.69,886
6,PA,13.39,946
7,RS,13.21,5344
8,RN,12.96,474
9,PR,12.62,4923


### Висновок:
1) Кртичніть результату (Великий обʼєм + велика затримка)
2) Штати з дуже великою затримкаю але з малим обʼємо: AC, AP, RR(Шум або гегографічна складність)
Можливі причіни:
- віддаленість
- слабка логістика
- мало складів
3) Затримки не тільки в віддалених регіонах,
але і в ключових штатах з великим обʼємом замовлень.
Можливі причини:
- процесів 
- перевізників
- складів

### Рекомендації:
1) Пріорітезувати великі штати
- SP
- RJ
- MG
- RS
- PR
2) Оптимізувати логістику, наприклад:
- нові склади
- інші перевізники
- оптимізація маршрутів
3) Розділити SLA по регіонах. 
4) Створити різні ETA для кожного штату
5) необіцяти занадто швидку доставку
6) попереджати клієнтів
7) Перевірити перевізників

In [34]:
# 24. Які штати мають найбільші затримки і в яких категоріях товарів ?
query = """
SELECT
    c.customer_state,
    (julianday(o.order_estimated_delivery_date)- julianday(o.order_delivered_customer_date)) AS avg_delay_days
FROM 
    orders o
JOIN customers c ON o.customer_id = c.customer_id
WHERE
    o.order_delivered_customer_date IS NOT NULL
    AND o.order_estimated_delivery_date IS NOT NULL
    AND c.customer_state = 'SP'
GROUP BY c.customer_state
ORDER BY avg_delay_days DESC;
"""
pd.read_sql_query(query, conn)


,customer_state,avg_delay_days
0,SP,10.558623


## Блок 7 — Аналіз продавців
25. Скільки замовлень у середнього продавця?
26. Хто топ-10 продавців за кількістю продажів?
27. Чи є залежність між рейтингом і продавцем?
28. Які продавці мають найбільше негативних відгуків?

In [35]:
# 25. Скільки замовлень у середнього продавця? 
query = """
SELECT
    AVG(count_orders) AS avg_order_salles
FROM
(
SELECT
    s.seller_id,
    COUNT(*) AS count_orders
FROM
    order_items oi
JOIN sellers s ON oi.seller_id = s.seller_id
GROUP BY s.seller_id
ORDER BY count_orders DESC
LIMIT 10
) 
"""
pd.read_sql_query(query, conn)


,avg_order_salles
0,1594.2


In [36]:
# 26. Хто топ-10 продавців за кількістю продажів?
query = """
SELECT
    s.seller_id,
    COUNT(*) AS count_orders
FROM
    order_items oi
JOIN sellers s ON oi.seller_id = s.seller_id
GROUP BY s.seller_id
ORDER BY count_orders DESC
LIMIT 10;
"""
pd.read_sql_query(query, conn)


,seller_id,count_orders
0,6560211a19b47992c3666cc44a7e94c0,2033
1,4a3ca9315b744ce9f8e9374361493884,1987
2,1f50f920176fa81dab994f9023523100,1931
3,cc419e0650a3c5ba77189a1882b7556a,1775
4,da8622b14eb17ae2831f4ac5b9dab84a,1551
5,955fee9216a65b617aa5c0531780ce60,1499
6,1025f0e2d44d7041d6cf58b6550e0bfa,1428
7,7c67e1448b00f6e969d365cea6b010ab,1364
8,ea8482cd71df3c1969d7b9473ff13abc,1203
9,7a67c85e85bb2ce8582c35f2203ad736,1171


In [37]:
# 27. Чи є залежність між рейтингом і продавцем?
query = """
SELECT
    MIN(avg_reviews_score) AS min_avg,
    MAX(avg_reviews_score) AS max_avg,
    MAX(avg_reviews_score) - MIN(avg_reviews_score) AS difference_avg
FROM
(
SELECT
    s.seller_id,
    COUNT(DISTINCT oi.order_id) AS count_orders,
    ROUND(AVG(orr.review_score), 3) AS avg_reviews_score,
    COUNT(orr.review_score) AS count_reviews
FROM
    order_items oi
JOIN sellers s ON oi.seller_id = s.seller_id
JOIN order_reviews orr ON orr.order_id = oi.order_id
GROUP BY s.seller_id
HAVING COUNT(DISTINCT oi.order_id) > 50
ORDER BY avg_reviews_score ASC
)
"""
pd.read_sql_query(query, conn)



,min_avg,max_avg,difference_avg
0,2.199,4.82,2.621


### Висконовок:
1) Є залежність між продавцями і рейтингом

### Рекомнедації:
1) Ввести рейтинг продавцівє
2) Фільтрувати поганих продавців
3) Ввести SLA для продавців
4) Показувати "Top Sellers"



In [38]:
# 28. Які продавці мають найбільше негативних відгуків?
query = """
SELECT
    s.seller_id,
    ROUND(AVG(orr.review_score), 3) AS avg_reviews_score,
    SUM(CASE WHEN orr.review_score <= 2  THEN 1 ELSE 0 END) AS count_reviews_1_and_2,
    COUNT(*) AS cnt_all
FROM
    order_items oi
JOIN sellers s ON oi.seller_id = s.seller_id
JOIN order_reviews orr ON orr.order_id = oi.order_id
GROUP BY s.seller_id
HAVING COUNT(DISTINCT oi.order_id) > 50
ORDER BY count_reviews_1_and_2 DESC
LIMIT 20;
"""


pd.read_sql_query(query, conn)


,seller_id,avg_reviews_score,count_reviews_1_and_2,cnt_all
0,7c67e1448b00f6e969d365cea6b010ab,3.348,402,1367
1,4a3ca9315b744ce9f8e9374361493884,3.804,392,1984
2,6560211a19b47992c3666cc44a7e94c0,3.909,361,2020
3,1f50f920176fa81dab994f9023523100,3.982,351,1932
4,1025f0e2d44d7041d6cf58b6550e0bfa,3.850,294,1431
5,cc419e0650a3c5ba77189a1882b7556a,4.070,272,1811
6,da8622b14eb17ae2831f4ac5b9dab84a,4.071,235,1568
7,955fee9216a65b617aa5c0531780ce60,4.052,215,1489
8,ea8482cd71df3c1969d7b9473ff13abc,3.953,199,1197
9,8b321bb669392f5163d04c59e235e066,3.995,174,1014


### Висновок:

### Рекомендації: 

## Блок 8 — Аналіз товарів
29. Які категорії товарів продаються найкраще?
30. Які категорії генерують найбільший дохід?
31. Які категорії мають найгірші відгуки?
32. Які категорії мають найдовшу доставку?

In [39]:
# 29. Які категорії товарів продаються найкраще?
query = """
SELECT
    ct.product_category_name_english,
    COUNT(DISTINCT oi.order_id) AS count_order
FROM
    products p
JOIN order_items oi ON p.product_id = oi.product_id
JOIN category_translation ct ON p.product_category_name = ct.product_category_name
WHERE ct.product_category_name_english IS NOT NULL
GROUP BY ct.product_category_name_english
ORDER BY count_order DESC
LIMIT 20
"""

pd.read_sql_query(query, conn)


,product_category_name_english,count_order
0,bed_bath_table,9417
1,health_beauty,8836
2,sports_leisure,7720
3,computers_accessories,6689
4,furniture_decor,6449
5,housewares,5884
6,watches_gifts,5624
7,telephony,4199
8,auto,3897
9,toys,3886


### Висновок:

### Рекомендації: 

In [40]:
# 30. Які категорії генерують найбільший дохід?
query = """
SELECT
    ct.product_category_name_english,
    COUNT(DISTINCT oi.order_id) AS count_orders,
    ROUND(SUM(oi.price + oi.freight_value), 2) AS revenue
FROM
    products p
JOIN order_items oi ON p.product_id = oi.product_id
JOIN category_translation ct ON p.product_category_name = ct.product_category_name
WHERE ct.product_category_name_english IS NOT NULL
GROUP BY ct.product_category_name_english
ORDER BY revenue DESC
LIMIT 20
"""
pd.read_sql_query(query, conn)


,product_category_name_english,count_orders,revenue
0,bed_bath_table,9417,4276366.70
1,health_beauty,8836,4115411.07
2,furniture_decor,6449,3762683.12
3,sports_leisure,7720,3646888.09
4,computers_accessories,6689,3475634.56
5,watches_gifts,5624,2728204.15
6,housewares,5884,2697480.73
7,telephony,4199,2520165.41
8,auto,3897,1758457.62
9,toys,3886,1741221.15


### Висновок:

### Рекомендації: 

In [41]:
# 31. Які категорії мають найгірші відгуки?
query = """
SELECT
    ct.product_category_name_english,
    ROUND(AVG(orr.review_score), 3) AS avg_review,
    COUNT(orr.review_score) AS count_score,
    SUM(CASE WHEN orr.review_score <= 2 THEN 1 ELSE 0 END) AS count_1_and_2,
    ROUND(100.0 * SUM(CASE WHEN orr.review_score <= 2 THEN 1 ELSE 0 END)/ COUNT(orr.review_score), 2) AS percent_negative_review
FROM
    products p
JOIN order_items oi ON p.product_id = oi.product_id
JOIN category_translation ct ON p.product_category_name = ct.product_category_name
JOIN order_reviews orr ON orr.order_id = oi.order_id
GROUP BY ct.product_category_name_english 
HAVING COUNT(orr.review_score) > 50
ORDER BY percent_negative_review DESC
LIMIT 20;
"""

pd.read_sql_query(query, conn)


,product_category_name_english,avg_review,count_score,count_1_and_2,percent_negative_review
0,fashion_male_clothing,3.641,131,37,28.24
1,office_furniture,3.493,1687,440,26.08
2,fixed_telephony,3.683,262,67,25.57
3,audio,3.825,361,79,21.88
4,home_confort,3.830,435,88,20.23
5,construction_tools_safety,3.845,193,39,20.21
6,furniture_decor,3.903,8331,1621,19.46
7,bed_bath_table,3.896,11137,2112,18.96
8,computers_accessories,3.931,7849,1461,18.61
9,art,3.937,207,38,18.36


### Висновок:

### Рекомендації: 

In [42]:
# 32. Які категорії мають найдовшу доставку?
query = """
SELECT
    ct.product_category_name_english,
    ROUND(AVG(julianday(order_delivered_customer_date) - julianday(order_approved_at)), 2) AS delivery_time,
    COUNT(*) AS count_order
FROM
    products p
JOIN order_items oi ON p.product_id = oi.product_id
JOIN category_translation ct ON p.product_category_name = ct.product_category_name
JOIN orders o ON o.order_id = oi.order_id
GROUP BY ct.product_category_name_english 
HAVING COUNT(*) > 50
ORDER BY delivery_time DESC
LIMIT 20;
"""

pd.read_sql_query(query, conn)


,product_category_name_english,delivery_time,count_order
0,office_furniture,20.25,1691
1,christmas_supplies,15.34,153
2,fashion_shoes,15.12,262
3,home_appliances_2,13.40,238
4,furniture_living_room,13.38,503
5,fashion_underwear_beach,13.35,131
6,garden_tools,13.22,4347
7,consoles_games,13.16,1137
8,home_confort,13.06,434
9,computers,12.91,203


### Висновок:

### Рекомендації: 

## Блок 9 — Географічний аналіз
33. В яких штатах найбільше покупців?
34. В яких штатах найбільше продавців?
35. Чи є різниця в середньому чеку між регіонами?
36. Чи є залежність між відстанню та часом доставки?

In [45]:
# 33. В яких штатах найбільше покупців?
query = """
SELECT
    customer_state,
    COUNT(DISTINCT customer_unique_id) AS count_coustomers
FROM
    customers
GROUP BY customer_state
ORDER BY count_coustomers DESC
LIMIT 10;
"""

pd.read_sql_query(query, conn)


,customer_state,count_coustomers
0,SP,40302
1,RJ,12384
2,MG,11259
3,RS,5277
4,PR,4882
5,SC,3534
6,BA,3277
7,DF,2075
8,ES,1964
9,GO,1952


### Висновок:

### Рекомендації: 

In [46]:
# 34. В яких штатах найбільше продавців?
query = """
SELECT
    seller_state,
    COUNT(DISTINCT seller_id) AS count_sellers
FROM
    sellers
GROUP BY seller_state
ORDER BY count_sellers DESC
LIMIT 10;
"""

pd.read_sql_query(query, conn)


,seller_state,count_sellers
0,SP,1849
1,PR,349
2,MG,244
3,SC,190
4,RJ,171
5,RS,129
6,GO,40
7,DF,30
8,ES,23
9,BA,19


### Висновок:

### Рекомендації: 

In [49]:
# 35. Чи є різниця в середньому чеку між регіонами?
query = """
SELECT
    c.customer_state,
    ROUND(AVG(order_total), 2) AS avg_order_value,
    COUNT(*) AS orders_count,
    COUNT(DISTINCT customer_unique_id) AS count_customers
FROM
(
    SELECT
        o.order_id,
        o.customer_id,
        SUM(oi.price + oi.freight_value) AS order_total
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    GROUP BY o.order_id, o.customer_id
) t
JOIN customers c ON t.customer_id = c.customer_id
GROUP BY c.customer_state
HAVING COUNT(*) > 100
ORDER BY avg_order_value DESC;
"""

pd.read_sql_query(query, conn)


,customer_state,avg_order_value,orders_count,count_customers
0,MS,659.41,709,688
1,DF,645.77,2125,2062
2,RS,622.62,5432,5249
3,GO,621.78,2007,1942
4,ES,615.86,2025,1956
5,SC,605.64,3612,3513
6,RJ,599.46,12762,12303
7,MG,586.26,11544,11178
8,PR,578.90,4998,4840
9,MT,490.36,903,873


### Висновок:

### Рекомендації: 

In [77]:
#❗❗❗❗❗❗❗❗❗❗❗❗❗❗❗❗❗❗❗
# 36. Чи є залежність між відстанню та часом доставки?
query = """
SELECT
    AVG(distance) AS avg_distance,
    AVG(delivery_time) AS avg_delivery_time
FROM
(
SELECT
    (
        ABS(gc.geolocation_lat - gs.geolocation_lat) +
        ABS(gc.geolocation_lng - gs.geolocation_lng)
    ) AS distance,
/*
    6371 * 2 * 
    ASIN(
        SQRT(
            POWER(SIN((RADIANS(gc.geolocation_lat - gs.geolocation_lat)) / 2), 2)
            +
            COS(RADIANS(gc.geolocation_lat))
            * COS(RADIANS(gs.geolocation_lat))
            *
            POWER(SIN((RADIANS(gc.geolocation_lng - gs.geolocation_lng)) / 2), 2)
        )
    ) AS distance_km
*/
FROM ...
    julianday(o.order_delivered_customer_date) 
    - julianday(o.order_approved_at) AS delivery_time
FROM orders o
JOIN order_items oi 
    ON o.order_id = oi.order_id
JOIN sellers s 
    ON oi.seller_id = s.seller_id
JOIN customers c 
    ON o.customer_id = c.customer_id
JOIN geolocation gc 
    ON c.customer_zip_code_prefix = gc.geolocation_zip_code_prefix
JOIN geolocation gs 
    ON s.seller_zip_code_prefix = gs.geolocation_zip_code_prefix
WHERE 
    o.order_delivered_customer_date IS NOT NULL
    AND o.order_approved_at IS NOT NULL
);0
"""

pd.read_sql_query(query, conn)


DatabaseError: Execution failed on sql '
SELECT
    AVG(distance) AS avg_distance,
    AVG(delivery_time) AS avg_delivery_time
FROM
(
SELECT
    (
        ABS(gc.geolocation_lat - gs.geolocation_lat) +
        ABS(gc.geolocation_lng - gs.geolocation_lng)
    ) AS distance,
/*
    6371 * 2 * 
    ASIN(
        SQRT(
            POWER(SIN((RADIANS(gc.geolocation_lat - gs.geolocation_lat)) / 2), 2)
            +
            COS(RADIANS(gc.geolocation_lat))
            * COS(RADIANS(gs.geolocation_lat))
            *
            POWER(SIN((RADIANS(gc.geolocation_lng - gs.geolocation_lng)) / 2), 2)
        )
    ) AS distance_km
*/
FROM ...
    julianday(o.order_delivered_customer_date) 
    - julianday(o.order_approved_at) AS delivery_time
FROM orders o
JOIN order_items oi 
    ON o.order_id = oi.order_id
JOIN sellers s 
    ON oi.seller_id = s.seller_id
JOIN customers c 
    ON o.customer_id = c.customer_id
JOIN geolocation gc 
    ON c.customer_zip_code_prefix = gc.geolocation_zip_code_p

In [54]:
# 36. Чи є залежність між відстанню та часом доставки?
query = """
SELECT
    CASE
        WHEN distance < 1 THEN 'short'
        WHEN distance < 5 THEN 'MEDIUM'
        ELSE 'long'
    END AS distance_group,

    ROUND(AVG(delivery_time), 2) AS avg_delivery_time,
    COUNT(*) AS orders_count
FROM
(
SELECT
    (
        ABS(gc.geolocation_lat - gs.geolocation_lat) +
        ABS(gc.geolocation_lng - gs.geolocation_lng)
    ) AS distance,
    julianday(o.order_delivered_customer_date) 
    - julianday(o.order_approved_at) AS delivery_time
FROM orders o
JOIN order_items oi 
    ON o.order_id = oi.order_id
JOIN sellers s 
    ON oi.seller_id = s.seller_id
JOIN customers c 
    ON o.customer_id = c.customer_id
JOIN geolocation gc 
    ON c.customer_zip_code_prefix = gc.geolocation_zip_code_prefix
JOIN geolocation gs 
    ON s.seller_zip_code_prefix = gs.geolocation_zip_code_prefix
WHERE 
    o.order_delivered_customer_date IS NOT NULL
    AND o.order_approved_at IS NOT NULL
)
GROUP BY distance_group
ORDER BY avg_delivery_time;0
"""

pd.read_sql_query(query, conn)


DatabaseError: Execution failed on sql '
SELECT
    CASE
        WHEN distance < 1 THEN 'short'
        WHEN distance < 5 THEN 'MEDIUM'
        ELSE 'long'
    END AS distance_group,

    ROUND(AVG(delivery_time), 2) AS avg_delivery_time,
    COUNT(*) AS orders_count
FROM
(
SELECT
    (
        ABS(gc.geolocation_lat - gs.geolocation_lat) +
        ABS(gc.geolocation_lng - gs.geolocation_lng)
    ) AS distance,
    julianday(o.order_delivered_customer_date) 
    - julianday(o.order_approved_at) AS delivery_time
FROM orders o
JOIN order_items oi 
    ON o.order_id = oi.order_id
JOIN sellers s 
    ON oi.seller_id = s.seller_id
JOIN customers c 
    ON o.customer_id = c.customer_id
JOIN geolocation gc 
    ON c.customer_zip_code_prefix = gc.geolocation_zip_code_prefix
JOIN geolocation gs 
    ON s.seller_zip_code_prefix = gs.geolocation_zip_code_prefix
WHERE 
    o.order_delivered_customer_date IS NOT NULL
    AND o.order_approved_at IS NOT NULL
)
GROUP BY distance_group
ORDER BY avg_del

### Висновок:

### Рекомендації: 

## Блок 10 — Ризики для бізнесу
37. Яка частка скасованих замовлень?
38. Чи пов’язані скасування з певними продавцями?
39. Чи впливає довга доставка на скасування?
40. Які категорії мають найбільше проблем?

In [80]:
# Назви таблиць:
print('Назви таблиць:')
for name, df in dataframes.items():
    print(name, df.columns)


Назви таблиць:
customers Index(['customer_id', 'customer_unique_id', 'customer_zip_code_prefix',
       'customer_city', 'customer_state'],
      dtype='object')
geolocation Index(['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng',
       'geolocation_city', 'geolocation_state'],
      dtype='object')
order_items Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value'],
      dtype='object')
order_payments Index(['order_id', 'payment_sequential', 'payment_type',
       'payment_installments', 'payment_value'],
      dtype='object')
order_reviews Index(['review_id', 'order_id', 'review_score', 'review_comment_title',
       'review_comment_message', 'review_creation_date',
       'review_answer_timestamp'],
      dtype='object')
orders Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date

In [81]:
# Зв'язки між таблицям:
# orders.customer_id → customers.customer_id
# order_items.order_id → orders.order_id
# order_items.product_id → products.product_id
# order_items.seller_id → sellers.seller_id
# order_payments.order_id → orders.order_id
# order_reviews.order_id → orders.order_id
# products.product_category_name → category_translation.product_category_name

In [120]:
# 37. Яка частка скасованих замовлень?
query = """
SELECT
    COUNT(*) AS count_all,
    SUM(CASE WHEN order_status = 'canceled' THEN 1 ELSE 0 END) AS count_canceled,
    SUM(100.0 * CASE WHEN order_status = 'canceled' THEN 1 ELSE 0 END) / COUNT(*) AS percent_canceled
FROM
    orders
"""

pd.read_sql_query(query, conn)


,count_all,count_canceled,percent_canceled
0,99441,625,0.628513


In [119]:
# 38. Чи пов’язані скасування з певними продавцями?
query = """
SELECT
    oi.seller_id,
    COUNT(DISTINCT CASE WHEN o.order_status = 'canceled' THEN o.order_id END) AS count_canceled
FROM
    order_items oi
JOIN orders o ON oi.order_id = o.order_id
GROUP BY oi.seller_id
HAVING count_canceled > 1
ORDER BY count_canceled DESC
LIMIT 20;
"""

pd.read_sql_query(query, conn)


,seller_id,count_canceled
0,cc419e0650a3c5ba77189a1882b7556a,9
1,6560211a19b47992c3666cc44a7e94c0,7
2,620c87c171fb2a6dd6e8bb4dec959fc6,6
3,c3867b4666c7d76867627c2f7fb22e21,5
4,a416b6a846a11724393025641d4edd5e,5
5,81783131d2a97c8d44d406a4be81b5d9,5
6,ffff564a4f9085cd26170f4732393726,4
7,dbc22125167c298ef99da25668e1011f,4
8,b2ba3715d723d245138f291a6fe42594,4
9,8444e55c1f13cd5c179851e5ca5ebd00,4


In [ ]:
# 39. Чи впливає довга доставка на скасування?
query = """
SELECT
    count_canceled,
    delivery_time
FROM
(
SELECT
    DISTINCT CASE WHEN order_status = 'canceled' THEN order_id END AS count_canceled,
    julianday(order_delivered_customer_date) - julianday(order_approved_at) AS delivery_time,
    
FROM
    orders
ORDER BY delivery_time DESC
)
WHERE count_canceled IS NOT NULL 
"""

pd.read_sql_query(query, conn)


,count_canceled,delivery_time
0,65d1e226dfaeb8cdc42f665422522d14,35.027512
1,2c45c33d2f9cb8ff8b1c86cc28c11c30,30.175706
2,1950d777989f6a877539f53795b4c3c3,30.047060
3,8beb59392e21af5eb9547ae1a9938d06,10.175845
4,dabf2b0e35b423f94618bf965fcb7514,7.041678
...,...,...
620,b159d0ce7cd881052da94fa165617b05,NaN
621,e49e7ce1471b4693482d40c2bd3ad196,NaN
622,6560fb10610771449cb0463c5ba12199,NaN
623,3a3cddda5a7c27851bd96c3313412840,NaN


In [ ]:
# 40. Які категорії мають найбільше проблем?

### Важливі слова:
1) Стимулює
2) Стабільний сегмент
3) Платіжоспроможність
4) Кореляція

1) ROW_NUMBER()
2) RANK()
3) LEAD()

### Додаткові питання:
1) Чи впливає кількість фото на продажі ?
2) Чи впливає довжина опису на продажі ?
3) Чи впливає вага товару на швидкість і вартість доставки ?
4) Аналіз (Repeat Purchase Rate) ?
- Low repead purchase rate
5) Як знайти snapshot клієнта ?
6) Скільки в середньому тварів у замовленні ?
7) 